In [2]:
import pandas as pd

In [4]:
reliance_df = pd.read_csv('../data/raw/RELIANCE_stock_data.csv')
infy_df     = pd.read_csv('../data/raw/INFY_stock_data.csv')
tcs_df      = pd.read_csv('../data/raw/TCS_stock_data.csv')
hdfcbank_df = pd.read_csv('../data/raw/HDFCBANK_stock_data.csv')

In [14]:
tickers = ['RELIANCE', 'INFY', 'TCS', 'HDFCBANK']

stock_df = pd.concat([reliance_df, infy_df, tcs_df, hdfcbank_df], keys=tickers).reset_index(level=0).rename(columns={'level_0': 'Ticker'})
stock_df

,Ticker,Date,Close,High,Low,Open,Volume
0,RELIANCE,2024-05-27,1455.477417,1473.990419,1450.811997,1469.349718,6629010
1,RELIANCE,2024-05-28,1445.501221,1467.637431,1442.076616,1457.214574,7820162
2,RELIANCE,2024-05-29,1430.189575,1447.287998,1427.881584,1435.872489,7383556
3,RELIANCE,2024-05-30,1414.381470,1429.817271,1409.666368,1424.953247,13206858
4,RELIANCE,2024-05-31,1419.890747,1431.653660,1411.800602,1420.784159,31069832
...,...,...,...,...,...,...,...
493,HDFCBANK,2026-05-21,759.150024,768.250000,755.150024,767.000000,34664725
494,HDFCBANK,2026-05-22,766.799988,775.000000,759.150024,759.150024,25604071
495,HDFCBANK,2026-05-25,786.849976,787.849976,775.200012,776.000000,26043952
496,HDFCBANK,2026-05-26,778.900024,790.849976,776.750000,784.049988,31528062


# Creating basic features

In [29]:
# Daily returns
stock_df['Daily_Return'] = stock_df.groupby('Ticker')['Close'].pct_change()
stock_df.fillna(0, inplace=True)

# Moving averages for 7, 14 and 30 days
stock_df['MA_7'] = stock_df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=7).mean())
stock_df['MA_14'] = stock_df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=14).mean())
stock_df['MA_30'] = stock_df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=30).mean())

# Volatility (standard deviation of daily returns) over a 30-day window
stock_df['Volatility_30'] = stock_df.groupby('Ticker')['Daily_Return'].transform(lambda x: x.rolling(window=30).std())

# Volume change
stock_df['Volume_Change'] = stock_df.groupby('Ticker')['Volume'].pct_change()

stock_df.head(50)

,Ticker,Date,Close,High,Low,Open,Volume,Daily_Return,MA_7,MA_14,MA_30,Volatility_30,Volume_Change,Price_Movement
0,RELIANCE,2024-05-27,1455.477417,1473.990419,1450.811997,1469.349718,6629010,0.000000,NaN,NaN,NaN,NaN,NaN,0
1,RELIANCE,2024-05-28,1445.501221,1467.637431,1442.076616,1457.214574,7820162,-0.006854,NaN,NaN,NaN,NaN,0.179688,0
2,RELIANCE,2024-05-29,1430.189575,1447.287998,1427.881584,1435.872489,7383556,-0.010593,NaN,NaN,NaN,NaN,-0.055831,0
3,RELIANCE,2024-05-30,1414.381470,1429.817271,1409.666368,1424.953247,13206858,-0.011053,NaN,NaN,NaN,NaN,0.788685,1
4,RELIANCE,2024-05-31,1419.890747,1431.653660,1411.800602,1420.784159,31069832,0.003895,NaN,NaN,NaN,NaN,1.352553,1
5,RELIANCE,2024-06-03,1499.228516,1503.372890,1448.280652,1472.104322,21527942,0.055876,NaN,NaN,NaN,NaN,-0.307111,0
6,RELIANCE,2024-06-04,1387.009155,1487.043779,1349.313184,1487.043779,36709098,-0.074851,1435.954014,NaN,NaN,NaN,0.705184,1
7,RELIANCE,2024-06-05,1410.311646,1420.635260,1373.930892,1411.552462,17464890,0.016801,1429.501761,NaN,NaN,NaN,-0.524235,1
8,RELIANCE,2024-06-06,1421.081909,1433.092985,1410.907217,1424.456953,17855722,0.007637,1426.013288,NaN,NaN,NaN,0.022378,1
9,RELIANCE,2024-06-07,1459.150024,1461.185011,1416.019306,1418.004612,18558696,0.026788,1430.150495,NaN,NaN,NaN,0.039370,1


# Creating target label

In [30]:
# Create a target variable for stock price movement (1 for up, 0 for down)
stock_df['Price_Movement'] = (stock_df.groupby('Ticker')['Close'].shift(-1) > stock_df['Close']).astype(int)

stock_df

,Ticker,Date,Close,High,Low,Open,Volume,Daily_Return,MA_7,MA_14,MA_30,Volatility_30,Volume_Change,Price_Movement
0,RELIANCE,2024-05-27,1455.477417,1473.990419,1450.811997,1469.349718,6629010,0.000000,NaN,NaN,NaN,NaN,NaN,0
1,RELIANCE,2024-05-28,1445.501221,1467.637431,1442.076616,1457.214574,7820162,-0.006854,NaN,NaN,NaN,NaN,0.179688,0
2,RELIANCE,2024-05-29,1430.189575,1447.287998,1427.881584,1435.872489,7383556,-0.010593,NaN,NaN,NaN,NaN,-0.055831,0
3,RELIANCE,2024-05-30,1414.381470,1429.817271,1409.666368,1424.953247,13206858,-0.011053,NaN,NaN,NaN,NaN,0.788685,1
4,RELIANCE,2024-05-31,1419.890747,1431.653660,1411.800602,1420.784159,31069832,0.003895,NaN,NaN,NaN,NaN,1.352553,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
493,HDFCBANK,2026-05-21,759.150024,768.250000,755.150024,767.000000,34664725,-0.000461,762.342861,769.689287,781.821670,0.014474,0.443554,1
494,HDFCBANK,2026-05-22,766.799988,775.000000,759.150024,759.150024,25604071,0.010077,764.800005,768.789285,780.791669,0.014122,-0.261380,1
495,HDFCBANK,2026-05-25,786.849976,787.849976,775.200012,776.000000,26043952,0.026148,767.271432,769.828570,780.010002,0.014668,0.017180,0
496,HDFCBANK,2026-05-26,778.900024,790.849976,776.750000,784.049988,31528062,-0.010104,768.900007,768.567858,779.483335,0.014365,0.210571,0


# Save dataframe as a CSV file

In [31]:
stock_df.to_csv('../data/processed/stock_features.csv', index=False)